# Exploratory Data Analysis

Data: Sparkov synthetic credit card transactions. Cleaned CSVs in `cleaned_data_files/`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy.stats import chi2_contingency
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['axes.prop_cycle'] = plt.cycler(color=['#3498db', '#c0392b'])
BLUE, RED = '#3498db', '#c0392b'
%matplotlib inline

## 1. Load data

In [ ]:
data_dir = Path('cleaned_data_files')
files = list(data_dir.rglob('*.csv'))
len(files)

In [ ]:
dfs = [pd.read_csv(f) for f in files]
df = pd.concat(dfs, ignore_index=True)
del dfs
df.shape

In [ ]:
df.duplicated().sum()
df['trans_datetime'] = pd.to_datetime(df['trans_datetime'])
df['trans_datetime'].min(), df['trans_datetime'].max()
df['amt'].quantile([0, 0.5, 0.95, 0.99, 1])
df['age'].min(), df['age'].max()
df.nunique()

In [ ]:
df['category'].value_counts()
df['gender'].value_counts()

In [ ]:
df.head()
df.info()

## 2. Missing values

In [ ]:
missing = df.isnull().sum()
missing[missing > 0]

## 3. Target (is_fraud)

Fraud is rare; handle with class weights or resampling.

In [ ]:
fraud_counts = df['is_fraud'].value_counts()
fraud_pct = df['is_fraud'].value_counts(normalize=True) * 100
pd.DataFrame({'count': fraud_counts, 'pct': fraud_pct})

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
fraud_counts.plot(kind='bar', ax=ax[0], color=[BLUE, RED])
ax[0].set_title('Fraud vs non-fraud count')
ax[0].set_xticklabels(['Non-fraud', 'Fraud'])
ax[0].set_ylabel('Count')
fraud_pct.plot(kind='pie', ax=ax[1], autopct='%1.2f%%', colors=[BLUE, RED])
ax[1].set_title('Fraud share (%)')
ax[1].set_ylabel('')
plt.tight_layout()
plt.show()

## 4. Univariate

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(12, 10))
df['amt'].hist(bins=80, ax=ax[0,0], color=BLUE, alpha=0.85)
ax[0,0].set_title('Transaction amount')
ax[0,0].set_ylabel('Count')
df['age'].hist(bins=30, ax=ax[0,1], color=BLUE, alpha=0.85)
ax[0,1].set_title('Age')
ax[0,1].set_ylabel('Count')
df['city_pop'].apply(lambda x: np.log10(x+1)).hist(bins=50, ax=ax[1,0], color=BLUE, alpha=0.85)
ax[1,0].set_title('City population (log10)')
ax[1,0].set_ylabel('Count')
df['category'].value_counts().head(15).plot(kind='barh', ax=ax[1,1], color=BLUE)
ax[1,1].set_title('Category (top 15)')
ax[1,1].set_xlabel('Count')
plt.tight_layout()
plt.show()

amt is right-skewed; most txns are small. 

## 5. Temporal

In [ ]:
df['trans_datetime'] = pd.to_datetime(df['trans_datetime'])
df['hour'] = df['trans_datetime'].dt.hour
df['day_of_week'] = df['trans_datetime'].dt.day_name()
df['month'] = df['trans_datetime'].dt.month
df['date'] = df['trans_datetime'].dt.date

Higher fraud % at night; some variation by day and month.

In [ ]:
fraud_by_hour = df.groupby('hour')['is_fraud'].mean() * 100
fraud_by_dow = df.groupby('day_of_week')['is_fraud'].mean() * 100
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
fraud_by_dow = fraud_by_dow.reindex([d for d in day_order if d in fraud_by_dow.index])
fraud_by_month = df.groupby('month')['is_fraud'].mean() * 100

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14, 4))
fraud_by_hour.plot(ax=ax[0], color=BLUE)
ax[0].set_title('Fraud % by hour')
ax[0].set_xlabel('Hour')
ax[0].set_ylabel('Fraud %')
fraud_by_dow.plot(kind='bar', ax=ax[1], color=BLUE)
ax[1].set_title('Fraud % by day of week')
ax[1].set_xlabel('Day')
ax[1].tick_params(axis='x', rotation=45)
fraud_by_month.plot(kind='bar', ax=ax[2], color=BLUE)
ax[2].set_title('Fraud % by month')
ax[2].set_xlabel('Month')
plt.tight_layout()
plt.show()

## 6. Haversine (customer–merchant)

In [ ]:
def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return R * c

df['haversine_km'] = haversine(df['lat'], df['long'], df['merch_lat'], df['merch_long'])

Distributions overlap; mean/median differ a bit. Use Haversine as a feature but it won’t separate on its own.

In [ ]:
df.groupby('is_fraud')['haversine_km'].agg(['mean','median','count'])

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for lab, sub in df.groupby('is_fraud'):
    sub['haversine_km'].clip(upper=sub['haversine_km'].quantile(0.99)).hist(ax=ax, bins=50, alpha=0.6, label=['Non-fraud','Fraud'][lab], color=[BLUE, RED][lab])
ax.set_title('Haversine distance (99th pct cap)')
ax.set_xlabel('km')
ax.set_ylabel('Count')
ax.legend()
plt.tight_layout()
plt.show()

## 7. Fraud by category and state

In [ ]:
fraud_by_cat = df.groupby('category')['is_fraud'].agg(['count','mean'])
fraud_by_cat.columns = ['n','fraud_pct']
fraud_by_cat['fraud_pct'] = fraud_by_cat['fraud_pct'] * 100
fraud_by_cat = fraud_by_cat.sort_values('fraud_pct', ascending=False)
fraud_by_cat

In [ ]:
fraud_by_cat['fraud_pct'].sort_values(ascending=True).plot(kind='barh', figsize=(8, 6), color=BLUE)
plt.title('Fraud % by category')
plt.xlabel('Fraud %')
plt.tight_layout()
plt.show()

In [ ]:
df.groupby('gender')['is_fraud'].mean() * 100
df.groupby('is_fraud')['age'].agg(['mean','median','count'])

shopping_net, misc_net and a few others have higher fraud %. State and job vary; encode them for the model.

In [ ]:
fraud_by_state = df.groupby('state')['is_fraud'].agg(['count','mean'])
fraud_by_state.columns = ['n','fraud_pct']
fraud_by_state['fraud_pct'] = fraud_by_state['fraud_pct'] * 100
fraud_by_state = fraud_by_state[fraud_by_state['n'] >= 5000].sort_values('fraud_pct', ascending=False)
fraud_by_state.head(15)

## 7b. Fraud by job

In [ ]:
fraud_by_job = df.groupby('job')['is_fraud'].agg(['count','mean'])
fraud_by_job.columns = ['n','fraud_pct']
fraud_by_job['fraud_pct'] = fraud_by_job['fraud_pct'] * 100
fraud_by_job = fraud_by_job[fraud_by_job['n'] >= 1000].sort_values('fraud_pct', ascending=False)
fraud_by_job.head(15)
fraud_by_job['fraud_pct'].head(12).plot(kind='barh', figsize=(8, 5), color=BLUE)
plt.title('Fraud % by job (top 12)')
plt.xlabel('Fraud %')
plt.tight_layout()
plt.show()

In [ ]:
fraud_by_state['fraud_pct'].head(15).plot(kind='barh', figsize=(8, 5), color=BLUE)
plt.title('Fraud % by state')
plt.xlabel('Fraud %')
plt.tight_layout()
plt.show()

Fraud amounts are higher on average; both groups have a long right tail. log(amt) or bins may help.

In [ ]:
df.groupby('is_fraud')['amt'].quantile([0.25, 0.5, 0.75, 0.9, 0.99]).unstack(level=0)

In [ ]:
top_cats = df['category'].value_counts().head(5).index
sub = df[df['category'].isin(top_cats)]
ct = pd.crosstab(sub['hour'], sub['category'], values=sub['is_fraud'], aggfunc='mean') * 100
ct

In [ ]:
sns.heatmap(ct, cmap='YlOrRd', fmt='.2f')
plt.title('Fraud % by hour and category')
plt.xlabel('category')
plt.ylabel('hour')
plt.tight_layout()
plt.show()

Fraud % depends on hour within category; hour×category could help.

## 8. Amount: fraud vs non-fraud

In [ ]:
df.groupby('is_fraud')['amt'].agg(['count','mean','median','std'])

In [ ]:
sns.boxplot(x=df['is_fraud'], y=df['amt'], palette=[BLUE, RED])
plt.gca().set_xticklabels(['Non-fraud', 'Fraud'])
plt.title('Amount by fraud')
plt.xlabel('is_fraud')
plt.ylabel('amt')
plt.show()

## 9. Correlations

In [ ]:
numeric_cols = ['amt','lat','long','merch_lat','merch_long','city_pop','age','haversine_km','is_fraud']
corr = df[numeric_cols].corr()
corr['is_fraud'].sort_values(ascending=False)

In [ ]:
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, square=True, vmin=-0.5, vmax=0.5)
plt.title('Correlation matrix')
plt.tight_layout()
plt.show()

amt correlates most with is_fraud; rest are weak. Lat/long and merch coords are redundant—use distance.

## 10. Feature selection and engineering

Rank numerics by |corr| with is_fraud, categoricals by Cramér's V. Job as proxy for education.

In [ ]:
target_corr = corr['is_fraud'].drop('is_fraud')
rank_numeric = target_corr.reindex(target_corr.abs().sort_values(ascending=False).index)
rank_numeric

In [ ]:
def cramer_v(contingency):
    n = contingency.sum().sum()
    chi2, _, _, _ = chi2_contingency(contingency)
    r, c = contingency.shape
    return np.sqrt(chi2 / (n * min(r - 1, c - 1))) if min(r, c) > 1 else 0

cat_cols = ['gender', 'category', 'job', 'state']
rank_cat = []
for col in cat_cols:
    tab = pd.crosstab(df[col], df['is_fraud'])
    chi2, p, dof, _ = chi2_contingency(tab)
    v = cramer_v(tab)
    rank_cat.append({'feature': col, 'chi2': chi2, 'pvalue': p, 'cramer_v': v})

rank_cat = pd.DataFrame(rank_cat).sort_values('cramer_v', ascending=False)
rank_cat

In [ ]:
top_jobs = df['job'].value_counts().head(15).index
sub = df[df['job'].isin(top_jobs)]
pd.crosstab(sub['job'], sub['is_fraud'], normalize='index').plot(kind='bar', stacked=True, color=[BLUE, RED], figsize=(10, 4))
plt.title('Fraud share by job (top 15 by count)')
plt.xlabel('job')
plt.legend(['Non-fraud', 'Fraud'])
plt.xticks(rotation=60)
plt.tight_layout()
plt.show()

In [ ]:
engineered = ['haversine_km', 'hour', 'day_of_week', 'month']
df[engineered + ['is_fraud']].head()